In [14]:
import numpy as np
import pandas as pd

In [24]:
import pandas as pd


def f(x: float) -> float:
    """Target function: x² - 4x - 9"""
    return x**3 - 4*x - 9


def bisect(a: float, b: float, max_err: float, max_itr: int) -> pd.DataFrame:
    """
    Bisection method to find root of f in [a, b].

    Args:
        a: Left bound
        b: Right bound
        max_err: Convergence tolerance
        max_itr: Maximum iterations allowed

    Returns:
        DataFrame with columns: iteration, midpoint, error

    Raises:
        ValueError: If f(a) and f(b) don't have opposite signs
    """
    if f(a) * f(b) >= 0:
        raise ValueError(
            f"f(a) and f(b) must have opposite signs. "
            f"Got f({a})={f(a):.4f}, f({b})={f(b):.4f}"
        )

    records = []

    for i in range(1, max_itr + 1):
        mid = (a + b) / 2
        error = abs(b - a)
        records.append({"iteration": i, "midpoint": mid, "error": error})

        if error < max_err:
            print(f"Root converged after {i} iteration(s).")
            break

        if f(a) * f(mid) < 0:
            b = mid
        else:
            a = mid
    else:
        print(f"Warning: Did not converge within {max_itr} iterations.")

    return pd.DataFrame(records)


def get_inputs() -> tuple[float, float, float, int]:
    """Prompt user for bisection parameters with basic validation."""
    while True:
        try:
            raw = input("Enter a, b, max_err, max_itr: ").split()
            if len(raw) != 4:
                raise ValueError("Exactly 4 values required.")
            a, b, max_err, max_itr = float(raw[0]), float(raw[1]), float(raw[2]), int(raw[3])
            if a >= b:
                raise ValueError("a must be less than b.")
            if max_err <= 0:
                raise ValueError("max_err must be positive.")
            if max_itr <= 0:
                raise ValueError("max_itr must be a positive integer.")
            return a, b, max_err, max_itr
        except ValueError as e:
            print(f"Invalid input: {e}. Please try again.")


def main():
    a, b, max_err, max_itr = get_inputs()
    print(f"\nRunning bisection on [{a}, {b}] | tolerance={max_err} | max_itr={max_itr}\n")

    try:
        results = bisect(a, b, max_err, max_itr)
    except ValueError as e:
        print(f"Error: {e}")
        return

    print(results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    print(f"\nBest estimate of root: {results['midpoint'].iloc[-1]:.6f}")


if __name__ == "__main__":
    main()


Running bisection on [2.0, 3.0] | tolerance=0.001 | max_itr=30

Root converged after 11 iteration(s).
 iteration  midpoint    error
         1  2.500000 1.000000
         2  2.750000 0.500000
         3  2.625000 0.250000
         4  2.687500 0.125000
         5  2.718750 0.062500
         6  2.703125 0.031250
         7  2.710938 0.015625
         8  2.707031 0.007812
         9  2.705078 0.003906
        10  2.706055 0.001953
        11  2.706543 0.000977

Best estimate of root: 2.706543


In [26]:
import pandas as pd


def f(x: float) -> float:
    """Target function: x² - 4x - 9"""
    return x**3 - 4*x - 9


def validate_bracket(a: float, b: float) -> None:
    """Check that f(a) and f(b) have opposite signs."""
    if f(a) * f(b) >= 0:
        raise ValueError(
            f"f(a) and f(b) must have opposite signs. "
            f"Got f({a})={f(a):.4f}, f({b})={f(b):.4f}"
        )


def bisect(a: float, b: float, max_err: float, max_itr: int) -> pd.DataFrame:
    """
    Find root of f(x) in [a, b] using the bisection method.

    Args:
        a       : Left bound of the interval
        b       : Right bound of the interval
        max_err : Convergence tolerance (stops when |b - a| < max_err)
        max_itr : Maximum number of iterations

    Returns:
        DataFrame with columns: iteration, midpoint, f(midpoint), error

    Raises:
        ValueError: If the bracket is invalid (no sign change)
    """
    validate_bracket(a, b)

    records = []

    for i in range(1, max_itr + 1):
        mid = (a + b) / 2
        f_mid = f(mid)
        error = abs(b - a)

        records.append({
            "iteration" : i,
            "midpoint"  : mid,
            "f(midpoint)": f_mid,
            "error"     : error,
        })

        # Convergence check
        if error < max_err:
            print(f"✔ Root converged after {i} iteration(s).")
            break

        # Narrow the bracket
        if f(a) * f_mid < 0:
            b = mid
        else:
            a = mid

    else:
        print(f"⚠ Warning: Did not converge within {max_itr} iterations.")

    return pd.DataFrame(records)


def get_inputs() -> tuple[float, float, float, int]:
    """
    Prompt the user for bisection parameters with full validation.

    Expected input format: a  b  max_err  max_itr
    Example             : 4  6  0.0001   50
    """
    while True:
        try:
            raw = input("Enter a, b, max_err, max_itr: ").split()

            if len(raw) != 4:
                raise ValueError("Exactly 4 values required.")

            a, b    = float(raw[0]), float(raw[1])
            max_err = float(raw[2])
            max_itr = int(raw[3])

            if a >= b:
                raise ValueError(f"'a' must be less than 'b'. Got a={a}, b={b}.")
            if max_err <= 0:
                raise ValueError(f"max_err must be positive. Got {max_err}.")
            if max_itr <= 0:
                raise ValueError(f"max_itr must be a positive integer. Got {max_itr}.")

            return a, b, max_err, max_itr

        except ValueError as e:
            print(f"  Invalid input — {e} Please try again.\n")


def display_results(df: pd.DataFrame) -> None:
    """Print the iteration table and final root estimate."""
    float_fmt = lambda x: f"{x:.6f}"
    print("\n" + "=" * 55)
    print(df.to_string(index=False, float_format=float_fmt))
    print("=" * 55)
    print(f"  Root estimate : {df['midpoint'].iloc[-1]:.6f}")
    print(f"  f(root)       : {df['f(midpoint)'].iloc[-1]:.2e}")
    print(f"  Final error   : {df['error'].iloc[-1]:.2e}")
    print(f"  Total itr     : {len(df)}")
    print("=" * 55)


def main() -> None:
    print("=" * 55)
    print("         BISECTION METHOD — ROOT FINDER")
    print("=" * 55)
    print(f"  Function: f(x) = x² - 4x - 9\n")

    a, b, max_err, max_itr = get_inputs()
    print(f"\n  Interval  : [{a}, {b}]")
    print(f"  Tolerance : {max_err}")
    print(f"  Max itr   : {max_itr}\n")

    try:
        results = bisect(a, b, max_err, max_itr)
        display_results(results)
    except ValueError as e:
        print(f"\n  Error: {e}")


if __name__ == "__main__":
    main()

         BISECTION METHOD — ROOT FINDER
  Function: f(x) = x² - 4x - 9




  Interval  : [2.0, 3.0]
  Tolerance : 1e-10
  Max itr   : 50

✔ Root converged after 35 iteration(s).

 iteration  midpoint  f(midpoint)    error
         1  2.500000    -3.375000 1.000000
         2  2.750000     0.796875 0.500000
         3  2.625000    -1.412109 0.250000
         4  2.687500    -0.339111 0.125000
         5  2.718750     0.220917 0.062500
         6  2.703125    -0.061077 0.031250
         7  2.710938     0.079423 0.015625
         8  2.707031     0.009049 0.007812
         9  2.705078    -0.026045 0.003906
        10  2.706055    -0.008506 0.001953
        11  2.706543     0.000270 0.000977
        12  2.706299    -0.004118 0.000488
        13  2.706421    -0.001924 0.000244
        14  2.706482    -0.000827 0.000122
        15  2.706512    -0.000279 0.000061
        16  2.706528    -0.000004 0.000031
        17  2.706535     0.000133 0.000015
        18  2.706532     0.000064 0.000008
        19  2.706530     0.000030 0.000004
        20  2.706529     0.000013 0

In [1]:
import pandas as pd
from typing import Callable


# ── 1. Function wrapper ───────────────────────────────────────────────────────

class MathFunction:
    """Wraps a callable and provides evaluation and bracket validation."""

    def __init__(self, func: Callable[[float], float], label: str = "f(x)") -> None:
        self._func  = func
        self.label  = label

    def evaluate(self, x: float) -> float:
        return self._func(x)

    def validate_bracket(self, a: float, b: float) -> None:
        fa, fb = self.evaluate(a), self.evaluate(b)
        if fa * fb >= 0:
            raise ValueError(
                f"No sign change on [{a}, {b}]. "
                f"Got f({a})={fa:.4f}, f({b})={fb:.4f}."
            )

    def __call__(self, x: float) -> float:
        return self.evaluate(x)

    def __repr__(self) -> str:
        return f"MathFunction(label='{self.label}')"


# ── 2. Result container ───────────────────────────────────────────────────────

class BisectionResult:
    """Stores and displays the result of a bisection run."""

    def __init__(self, records: list[dict], converged: bool) -> None:
        self.df        = pd.DataFrame(records)
        self.converged = converged

    @property
    def root(self) -> float:
        return self.df["midpoint"].iloc[-1]

    @property
    def f_root(self) -> float:
        return self.df["f(midpoint)"].iloc[-1]

    @property
    def final_error(self) -> float:
        return self.df["error"].iloc[-1]

    @property
    def total_iterations(self) -> int:
        return len(self.df)

    def display(self) -> None:
        float_fmt = lambda x: f"{x:.6f}"
        sep = "=" * 58

        print(f"\n{sep}")
        print(self.df.to_string(index=False, float_format=float_fmt))
        print(sep)
        print(f"  Root estimate    : {self.root:.6f}")
        print(f"  f(root)          : {self.f_root:.2e}")
        print(f"  Final error      : {self.final_error:.2e}")
        print(f"  Total iterations : {self.total_iterations}")
        print(f"  Converged        : {'Yes' if self.converged else 'No'}")
        print(sep)

    def __repr__(self) -> str:
        return (f"BisectionResult(root={self.root:.6f}, "
                f"iterations={self.total_iterations}, "
                f"converged={self.converged})")


# ── 3. Solver ─────────────────────────────────────────────────────────────────

class BisectionSolver:
    """
    Finds a root of a MathFunction on [a, b] using the bisection method.

    Args:
        func    : A MathFunction instance
        a       : Left bound of the bracket
        b       : Right bound of the bracket
        max_err : Convergence tolerance
        max_itr : Maximum number of iterations
    """

    def __init__(
        self,
        func   : MathFunction,
        a      : float,
        b      : float,
        max_err: float,
        max_itr: int,
    ) -> None:
        self._validate_params(a, b, max_err, max_itr)
        self.func    = func
        self.a       = a
        self.b       = b
        self.max_err = max_err
        self.max_itr = max_itr
        self._result : BisectionResult | None = None

    # ── public ────────────────────────────────────────────────────────────────

    def solve(self) -> BisectionResult:
        """Run the bisection algorithm and return a BisectionResult."""
        self.func.validate_bracket(self.a, self.b)

        a, b    = self.a, self.b
        records = []
        converged = False

        for i in range(1, self.max_itr + 1):
            mid   = (a + b) / 2
            f_mid = self.func(mid)
            error = abs(b - a)

            records.append({
                "iteration"  : i,
                "midpoint"   : mid,
                "f(midpoint)": f_mid,
                "error"      : error,
            })

            if error < self.max_err:
                print(f"✔ Converged after {i} iteration(s).")
                converged = True
                break

            if self.func(a) * f_mid < 0:
                b = mid
            else:
                a = mid
        else:
            print(f"⚠ Did not converge within {self.max_itr} iterations.")

        self._result = BisectionResult(records, converged)
        return self._result

    @property
    def result(self) -> BisectionResult | None:
        """Returns the last result, or None if solve() hasn't been called."""
        return self._result

    # ── private ───────────────────────────────────────────────────────────────

    @staticmethod
    def _validate_params(a, b, max_err, max_itr) -> None:
        if a >= b:
            raise ValueError(f"'a' must be less than 'b'. Got a={a}, b={b}.")
        if max_err <= 0:
            raise ValueError(f"max_err must be positive. Got {max_err}.")
        if max_itr <= 0:
            raise ValueError(f"max_itr must be a positive integer. Got {max_itr}.")

    def __repr__(self) -> str:
        return (f"BisectionSolver(func={self.func!r}, "
                f"a={self.a}, b={self.b}, "
                f"max_err={self.max_err}, max_itr={self.max_itr})")


# ── 4. Input handler ──────────────────────────────────────────────────────────

class InputHandler:
    """Handles all user input and validation for the bisection solver."""

    PROMPT = "Enter a, b, max_err, max_itr: "

    @classmethod
    def get_inputs(cls) -> tuple[float, float, float, int]:
        while True:
            try:
                raw = input(cls.PROMPT).split()

                if len(raw) != 4:
                    raise ValueError("Exactly 4 values required.")

                a, b    = float(raw[0]), float(raw[1])
                max_err = float(raw[2])
                max_itr = int(raw[3])

                BisectionSolver._validate_params(a, b, max_err, max_itr)
                return a, b, max_err, max_itr

            except ValueError as e:
                print(f"  Invalid input — {e} Please try again.\n")


# ── 5. Application entry point ────────────────────────────────────────────────

class App:
    """Orchestrates the bisection application."""

    HEADER = "=" * 58

    def __init__(self) -> None:
        self.func = MathFunction(
            func  = lambda x: x**3 - 4*x - 9,
            label = "x² - 4x - 9",
        )

    def run(self) -> None:
        self._print_header()

        a, b, max_err, max_itr = InputHandler.get_inputs()
        self._print_config(a, b, max_err, max_itr)

        try:
            solver = BisectionSolver(self.func, a, b, max_err, max_itr)
            result = solver.solve()
            result.display()
        except ValueError as e:
            print(f"\n  Error: {e}")

    def _print_header(self) -> None:
        print(self.HEADER)
        print("        BISECTION METHOD — ROOT FINDER")
        print(self.HEADER)
        print(f"  Function : f(x) = {self.func.label}\n")

    def _print_config(self, a, b, max_err, max_itr) -> None:
        print(f"\n  Interval  : [{a}, {b}]")
        print(f"  Tolerance : {max_err}")
        print(f"  Max itr   : {max_itr}\n")


if __name__ == "__main__":
    App().run()

### Class design breakdown
# ```
# MathFunction          → wraps any callable, handles evaluation & bracket check
# BisectionResult       → stores the DataFrame, exposes clean properties
# BisectionSolver       → runs the algorithm, owns the math logic
# InputHandler          → all user I/O and validation in one place
# App                   → wires everything together, entry point

        BISECTION METHOD — ROOT FINDER
  Function : f(x) = x² - 4x - 9


  Interval  : [2.0, 3.0]
  Tolerance : 0.0001
  Max itr   : 30

✔ Converged after 15 iteration(s).

 iteration  midpoint  f(midpoint)    error
         1  2.500000    -3.375000 1.000000
         2  2.750000     0.796875 0.500000
         3  2.625000    -1.412109 0.250000
         4  2.687500    -0.339111 0.125000
         5  2.718750     0.220917 0.062500
         6  2.703125    -0.061077 0.031250
         7  2.710938     0.079423 0.015625
         8  2.707031     0.009049 0.007812
         9  2.705078    -0.026045 0.003906
        10  2.706055    -0.008506 0.001953
        11  2.706543     0.000270 0.000977
        12  2.706299    -0.004118 0.000488
        13  2.706421    -0.001924 0.000244
        14  2.706482    -0.000827 0.000122
        15  2.706512    -0.000279 0.000061
  Root estimate    : 2.706512
  f(root)          : -2.79e-04
  Final error      : 6.10e-05
  Total iterations : 15
  Converged        : Yes